In [ ]:
#!/usr/bin/env python3
import numpy as np
import os,sys,glob
from pathlib import Path
import tqdm
import logging
from helper import (filterObjects,getModelInfo,saveOutput, \
                    overlapRemoval, minDphilist, eff_trigger, \
                    getLLPDecayRadius,getLLPDecayTime,electronPtSmear,\
                    eff_track_EWK,eff_track_Strong, cutFlow)
from numpy import ndarray
from typing import Any, Dict, List, Tuple, Union
import multiprocessing
import subprocess
from computeEfficiencies import getObjects,preSelection

FORMAT = '%(levelname)s: %(message)s'
logging.basicConfig(format=FORMAT,datefmt='%m/%d/%Y %I:%M:%S %p')
logger = logging.getLogger()   

# Fix seed so results are reproducible!
np.random.seed(seed=123)

DelphesLLP_path = Path(os.path.abspath("./DelphesLLP"))
os.environ['ROOT_INCLUDE_PATH'] = os.path.join(DelphesLLP_path,"external")

import ROOT
ROOT.gSystem.Load(os.path.join(DelphesLLP_path,"libDelphes.so"))
ROOT.gInterpreter.Declare('#include "classes/SortableObject.h"')
ROOT.gInterpreter.Declare('#include "classes/DelphesClasses.h"')
ROOT.gInterpreter.Declare('#include "external/ExRootAnalysis/ExRootTreeReader.h"')
from ROOT import TFile,Electron, Jet, MissingET, Muon, TTree


# Define SRs and Cutflow
ewk_cutflow = cutFlow(name='EWK_cutflow',levels=['All', 'GRL and Cleaning', 'MET Trigger', 'Lepton Veto', 
                    'MET > 200 GeV', 'Jet pT > 100 GeV', 'min(DeltaPhi(JetMET)) > 1.0'])
strong_cutflow = cutFlow(name='Strong_cutflow',levels=['All', 'GRL and Cleaning', 'MET Trigger', 'Lepton Veto',
                      'MET > 250 GeV', 'Jet pT > 100,20,20 GeV', 'min(DeltaPhi(JetMET)) > 0.4'])
ewk_SR = cutFlow(name='EWK_SR',levels=['All', 'Kinematic', 'Tracklet Emulation', 'Leading tracklet',
                                'DeltaR(jet) > 0.4', 'DeltaR(electron) > 0.4', 'DeltaR(muon) > 0.4',
                                 '0.1 < Eta < 1.9'])
strong_SR = cutFlow(name='Strong_SR',levels=['All', 'Kinematic', 'Tracklet Emulation', 'Leading tracklet',
                                'DeltaR(jet) > 0.4', 'DeltaR(electron) > 0.4', 'DeltaR(muon) > 0.4',
                                 '0.1 < Eta < 1.9'])

In [ ]:
tauList  = [0.3,0.01]
inputFile = './run_113/wino_100GeV_0.300ns_delphes_events.root'
ijob = -1
tau0 = 0.3

In [ ]:
tauList = np.array(tauList)
eff_SR = cutFlow(name="Efficiencies",levels=['EWK SR', 'Strong SR'],
                    zero_weight=np.zeros(len(tauList)))

f = TFile(inputFile,'read')
DelphesTree = f.Get('Delphes')
nevts = DelphesTree.GetEntries()

totalweight = 0
ct=0

llp_effs = []
ewk_weights = []
llp_Rs = []
llp_pTs = []


disable = False
if ijob < 0:
    disable = True
for entry in tqdm.tqdm(range(nevts),position=ijob,
                        desc=inputFile,
                        leave=False,
                        disable=disable):
    DelphesTree.GetEntry(entry)
    ct+=1
    # weights = float(DelphesTree.Weight.At(0).Weight)
    weight = 1.0
    totalweight += weight
    llps,muons,electrons,jets,met = getObjects(DelphesTree)
    

    # # Reset cutflows to beginning
    ewk_cutflow.reset()
    strong_cutflow.reset()
    ewk_SR.reset()
    strong_SR.reset()

    # # Fill first key (All)
    ewk_cutflow.fill(weight)
    strong_cutflow.fill(weight)
    ewk_SR.fill(weight)
    strong_SR.fill(weight)

    preSel_eff_EWK,preSel_eff_Strong = preSelection(muons,electrons,jets,met,weight,
                                                    ewk_cutflow,strong_cutflow)

    if (not preSel_eff_EWK):
        continue

    if not llps:
        continue

    # Compute relevant LLP variables
    for llp in llps:
        llp.daughter = DelphesTree.bsmDirectDaughters.At(llp.D1)
        llp.decayR = getLLPDecayRadius(llp)
        llp.decayT = getLLPDecayTime(llp)
        llp.gamma = llp.P4().Gamma()
        llp.smearedPt = electronPtSmear(llp.PT, llp.Charge)
        track_eff_EWK =  eff_track_EWK.efficiency(llp.Eta,llp.decayR)
        if np.isnan(track_eff_EWK):
            track_eff_EWK = 0.0
        track_eff_Strong =  eff_track_Strong.efficiency(llp.Eta,llp.decayR)
        if np.isnan(track_eff_Strong):
            track_eff_Strong = 0.0
            
        llp.tracklet_eff_EWK = track_eff_EWK
        llp.tracklet_eff_Strong = track_eff_Strong
        # Lifetime reweighting:
        if tau0 > 0.0:
            llp.lifetime_reweight = (tau0/tauList)*np.exp(-(llp.decayT/llp.gamma)*(1/tauList-1/tau0))
        else:
            llp.lifetime_reweight = np.ones(tauList.shape)

    R_values = [0.0,0.0,0.0,0.0]
    effs = [0.0,0.0]
    pts = [0.0,0.0]
    for llp in llps:
        if llp.PID == 1000024:
            illp = 0
        elif llp.PID == -1000024:
            illp = 1
        else:
            print(f'wrong LLP ID = {llp.PID}')
            raise ValueError()
        R_values[illp] = llp.decayR
        R_values[illp+2] = llp.decayT/llp.gamma
        effs[illp] = llp.tracklet_eff_EWK*llp.lifetime_reweight
        pts[illp] = llp.PT

    # Sort by smearedPt:
    llps = sorted(llps, key=lambda llp: llp.smearedPt,reverse=True)

    # Add one entry for each llp
    ewk_SR.fill_next(weight*preSel_eff_EWK*len(llps))
    
    # Selected llps with track effciency > 0:
    llps_EWK = [llp for llp in llps if llp.tracklet_eff_EWK > 0.0]
    fill_EWK = 0.0
    if not llps_EWK:
        continue
   
    
    fill_EWK = weight*preSel_eff_EWK*llps_EWK[0].tracklet_eff_EWK
    ewk_SR.fill_next(fill_EWK)
    

    # Select llps with smearedPt > minPT:
    # minPT = 60 (20) GeV for the model-independent (model-dependent) search strategy
    minPT = 60.0
    llps_EWK = [llp for llp in llps_EWK[:] if llp.smearedPt > minPT]
    if not llps_EWK:
        continue
    ewk_SR.fill_next(fill_EWK)
    

    # Remove LLPs with overlap to jets, electrons and muons:
    for objList in [jets,electrons,muons]:
        llps_EWK = overlapRemoval(llps_EWK,objList,0.4)
        if llps_EWK:
            ewk_SR.fill_next(fill_EWK)
    
    # Apply eta cut: 0.1 < abs(eta) < 1.9
    llps_EWK = [llp for llp in llps_EWK if 0.1 < abs(llp.Eta) < 1.9]
    if not llps_EWK:
        continue
    
    
    # Finally compute event weight:
    # evt_weight = weight*preSelectionEff*llp_eff    

    evt_weight_EWK = np.zeros(len(tauList))
    evt_weight_Strong = np.zeros(len(tauList))

    if not llps_EWK:
        continue
    # Require at least one LLP to be reconstructed and isolated
    llp_eff = 1.0 - np.prod([(1.0-llp.tracklet_eff_EWK*llp.lifetime_reweight)
                            for llp in llps_EWK],axis=0)
    llp_effs.append(llp_eff)
    llp_Rs.append(R_values)
    # llp_effs.append(effs)
    if llp_effs[-1][1] > 1e-2:
        for llp in llps_EWK:
            print(llp.PID,llp.PT,llp.gamma)

    
    evt_weight_EWK = weight*preSel_eff_EWK*llp_eff
    eff_SR.fill_level('EWK SR',evt_weight_EWK)
    
#End of loop
logger.info(f"Loop Ended! Evts analysed: {ct}")
eff_SR.divide(totalweight)
eff_dict = {}
eff_dict['Eff SR'] = eff_SR
eff_dict['totalweight'] = totalweight
eff_dict['Nevents'] = ct
eff_dict['inputFile'] = inputFile
eff_dict['tau_ns'] = tauList
eff_dict['tau0_ns'] = tau0

In [ ]:
print(eff_SR.to_string())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LogNorm

plt.rcParams.update({
    "text.usetex": True,
    "font.family": "sans-serif",
    "font.sans-serif": ["Helvetica"]})

plt.rcParams.update({"savefig.dpi" : 300}) #Figure resolution


#Define plotting style:
sns.set_style('ticks',{'font.family':'Times New Roman', 'font.serif':'Times New Roman'})
sns.set_context('paper', font_scale=1.8)
cm = plt.colormaps['RdYlBu']
cm_rev = plt.colormaps['RdYlBu_r']

In [ ]:
# ewk_weights = np.array(ewk_weights)
# llp_effs = np.array(llp_effs)
llp_Rs = np.array(llp_Rs)
# print(ewk_weights.shape,llp_effs.shape)

In [ ]:
plt.hist(llp_Rs[:,0],label=r'$R0$',bins=np.arange(50.,500.,25.),histtype='step')
plt.hist(llp_Rs[:,1],label=r'$R1$',bins=np.arange(50.,500.,25.),histtype='step')
plt.legend()
# plt.xscale('log')
plt.yscale('log')
plt.show()


In [ ]:
plt.hist(llp_Rs[:,2],label=r'$t0$',bins=np.arange(0.,2.,0.05),histtype='step')
plt.hist(llp_Rs[:,3],label=r'$t1$',bins=np.arange(0.,2.,0.05),histtype='step')
plt.vlines(x=0.3,ymin=1,ymax=1e2,colors='black')
plt.legend()
# plt.xscale('log')
plt.yscale('log')
plt.show()


In [ ]:
plt.hist(llp_Rs[:,0],label=r'$R0, \tau_0$',bins=np.arange(100.,500.,25.),weights=llp_effs[:,0])
plt.hist(llp_Rs[:,0],label=r'$R0, \tau_1$',bins=np.arange(100.,500.,25.),weights=llp_effs[:,1])
plt.legend()
# plt.xscale('log')
plt.yscale('log')
plt.show()


In [ ]:
plt.scatter(llp_Rs[:,0],llp_effs[:,0],label=r'$R0, \tau_0$')
plt.scatter(llp_Rs[:,0],llp_effs[:,1],label=r'$R0, \tau_1$')
plt.legend()
# plt.xscale('log')
plt.yscale('log')
plt.xlim(100,500)
plt.ylim(1e-4,1)
plt.show()


In [ ]:
plt.scatter(llp_Rs[:,2],llp_effs[:,0],label=r'$R0, \tau_0$')
plt.scatter(llp_Rs[:,2],llp_effs[:,1],label=r'$R0, \tau_1$')
plt.legend()
# plt.xscale('log')
plt.yscale('log')
plt.ylim(1e-7,1)
plt.xlim(1e-2,1.0)
plt.show()


In [ ]:
plt.hist(ewk_weights,bins=10)
plt.show()

In [ ]:
for i in range(llp_effs.shape[1]):
    plt.hist(llp_effs[:,i],label=r'$\tau = %1.3f$' %tauList[i],bins=np.logspace(-10,1,100))
plt.legend()
plt.xscale('log')
plt.yscale('log')
plt.grid()
plt.show()
